# Does the t-sampler cause the conditional-mean collapse?

`models/jit_dinov3/final.pt` is a **deterministic function of `h`**.  Sampling k
images from one representation returns k byte-identical images: within-`h` mean
pixel difference is 2e-5, roughly 200x below the 1/255 quantisation floor.  This
holds for EMA weights, raw weights, and the 55k checkpoint, and injecting noise
mid-trajectory at eta=3.0 does not change it.

That breaks E2 outright — "sample k images from one `h` and see what varies" has
nothing to measure — and it explains E1's mediocre fidelity (MRR 0.356, median
rank 5.5) as the same root cause: the model learned `E[x|h]`, the conditional
mean, rather than the conditional distribution.

## The hypothesis under test

Training draws `t ~ sigmoid(N(mu, sigma))` with `mu = -0.8` (JiT Tab. 3), where
`z_t = t*x + (1-t)*eps`, so t=0 is pure noise and t=1 is the clean image.  That
setting puts **50% of training mass below t=0.31 and only 2% above t=0.7**.

At low t the input carries no information about `x`, so the MSE-optimal
prediction is exactly `E[x|h]` — the model is *rewarded* for ignoring its noise.
And once `x_pred ≈ mu(h)` independent of `z`, the sampling ODE has the closed
form

    z(t) = mu(h) + (1-t)*(z_0 - mu(h))

which annihilates the initial noise linearly.  Every trajectory converges before
the model reaches the region (t >~ 0.5) where it does respond to its input.

**Hypothesis:** raising `mu` moves training mass toward the clean end, the model
learns to use `z`, and within-`h` diversity returns.

## Design

Three arms, identical in every respect except `--t_mu`:

| arm | t_mu | median t | mass above t=0.7 | role |
|---|---|---|---|---|
| A | -0.8 | 0.31 | 2% | control — current setting |
| B | 0.0 | 0.50 | 14% | test — SD3 default |
| C | +0.8 | 0.69 | 41% | test — strong clean-end emphasis |

**The control arm is not optional.** An undertrained model may sample diversely
simply because it has not yet collapsed, so a test arm alone cannot distinguish
"mu fixed it" from "fewer steps fixed it".  Arm A trained for the identical
budget is what rules that out.

Three outcomes and what each means:

- **A collapses, B/C do not** → hypothesis confirmed, `t_mu` is the cause.
- **All three collapse** → the t-sampler is not the lever; suspect dataset size
  (972 images x ~3300 epochs invites memorisation) or model capacity.
- **All three sample diversely** → collapse is a function of training *length*,
  not `t_mu`.  Re-run with a longer budget before concluding anything.

## Budget

**100k steps per arm — the same budget as the production run**, so arm A is a
true replication of `models/jit_dinov3/final.pt` and must reproduce its collapse.
A shorter budget would leave "the model has not collapsed *yet*" as a live
alternative explanation for any diversity seen in B and C.

At ~2000 steps/min on an A100 that is **~50 min per arm, ~2.5 h total**.  Arms
are checkpointed to Drive as they complete, so a disconnect costs at most the
arm in flight — re-running the training cell picks up from the last finished arm.
Drop arm C if you want to halve the wall clock; A vs B is the minimum test.

## 0 — Runtime check

Runtime → Change runtime type → **A100 GPU**.

In [ ]:
import torch
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU     : {name}  ({mem:.0f} GB)")
    BATCH_SIZE = 256 if mem >= 70 else 128 if mem >= 38 else 64
    print(f"-> batch_size = {BATCH_SIZE}")
else:
    BATCH_SIZE = 8
    raise SystemExit("No GPU. Runtime -> Change runtime type -> A100.")

## 1 — Mount Drive

Needs `MyDrive/jit_rcdm/train_packed.pt` (from `data/scripts/pack_dataset.py`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = "/content/drive/MyDrive/jit_rcdm"
PACKED    = f"{DRIVE_DIR}/train_packed.pt"
assert os.path.exists(PACKED), f"Missing {PACKED} — build it with data/scripts/pack_dataset.py"
print(f"packed dataset: {os.path.getsize(PACKED)/1e6:.0f} MB")

## 2 — Clone the repo

In [ ]:
import os, sys

REPO_DIR = "/content/jit_rcdm"
REPO_URL = "https://github.com/SeverinLe/master_implementation.git"
BRANCH   = "repo-restructure"     # branch carrying the --t_mu flag

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git -C {REPO_DIR} fetch --all -q && git -C {REPO_DIR} checkout -q {BRANCH} && git -C {REPO_DIR} pull -q
else:
    !git clone -q --branch {BRANCH} --single-branch {REPO_URL} {REPO_DIR}

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

# The flag must exist, or every arm silently trains at the default mu.
!git -C {REPO_DIR} log --oneline -1
assert "--t_mu" in open("scripts/train.py").read(), \
    "This branch predates the --t_mu flag; push the t-sampler change first."
print("OK — --t_mu available")

In [ ]:
!pip install -q transformers safetensors
print("dependencies installed")

## 3 — Confirm the t distribution before spending GPU time

Sanity check that `--t_mu` shifts the mass where the hypothesis says it does.
This is cheap and catches a mis-wired flag before three training runs, not after.

In [ ]:
import numpy as np, torch

def t_stats(mu, sigma=0.8, n=2_000_000):
    t = torch.sigmoid(mu + sigma * torch.randn(n)).numpy()
    return np.median(t), (t < 0.31).mean(), (t > 0.7).mean()

print(f"{'t_mu':>6} {'median t':>10} {'P(t<0.31)':>11} {'P(t>0.7)':>10}")
for mu in (-0.8, 0.0, 0.8):
    med, lo, hi = t_stats(mu)
    print(f"{mu:>6.1f} {med:>10.3f} {lo:>10.1%} {hi:>9.1%}")

## 4 — Train the three arms

Identical seed, identical budget, one variable.  `--no_wandb` keeps the ablation
self-contained; drop it to log curves.

**Restartable.**  Training writes to fast local disk, and each finished arm is
copied to Drive.  Re-running this cell after a disconnect skips completed arms
and resumes an interrupted one from its last 25k checkpoint, so the worst case
is losing < 25k steps rather than the session.

In [ ]:
TOTAL_STEPS = 100000    # matches the production run — see the budget note above
SAVE_EVERY  = 25000     # intermediate checkpoints, for resume after a disconnect
SEED_ARGS   = f"--batch_size {BATCH_SIZE} --grad_accum 1 --lr 1e-4 --warmup_steps 1000"
ARMS        = {"A_mu-0.8": -0.8, "B_mu0.0": 0.0, "C_mu0.8": 0.8}

import os, time, glob, shutil

LOCAL_DIR = "/content/ablation"                       # fast, ephemeral
ABL_DIR   = f"{DRIVE_DIR}/ablation_t_sampler"         # persistent
os.makedirs(LOCAL_DIR, exist_ok=True)
os.makedirs(ABL_DIR, exist_ok=True)

FINAL = "jit_rcdm_final.pt"        # train.py's naming — not "final.pt"


def local_final(name):
    return f"{LOCAL_DIR}/{name}/{FINAL}"


def restore_from_drive(name):
    # Bring a previously completed arm back after a session restart.
    src = f"{ABL_DIR}/{name}_{FINAL}"
    if os.path.exists(src) and not os.path.exists(local_final(name)):
        os.makedirs(f"{LOCAL_DIR}/{name}", exist_ok=True)
        shutil.copy(src, local_final(name))
        print(f"[{name}] restored from Drive")
    return os.path.exists(local_final(name))


for name, mu in ARMS.items():
    if restore_from_drive(name):
        print(f"[{name}] already complete — skipping")
        continue

    out = f"{LOCAL_DIR}/{name}"
    os.makedirs(out, exist_ok=True)
    partial = sorted(glob.glob(f"{out}/jit_rcdm_step*.pt"))
    resume = f" --resume {partial[-1]}" if partial else ""
    if resume:
        print(f"[{name}] resuming from {os.path.basename(partial[-1])}")

    print(f"\n{'='*70}\n[{name}] training with t_mu={mu}\n{'='*70}")
    t0 = time.time()
    # Built as one line: a backslash-continued "!" command inside an indented
    # block is not reliably parsed by IPython.
    cmd = (f"python scripts/train.py --encoder dinov3 --reps_file {PACKED} "
           f"--save_dir {out} --model S16 --image_size 224 "
           f"--t_mu {mu} --t_sigma 0.8 "
           f"--total_steps {TOTAL_STEPS} --save_interval {SAVE_EVERY} "
           f"--cfg_dropout 0.1 --no_wandb --device cuda {SEED_ARGS}{resume}")
    !{cmd}

    if os.path.exists(local_final(name)):
        shutil.copy(local_final(name), f"{ABL_DIR}/{name}_{FINAL}")
        # Intermediates have served their purpose; ~400 MB each.
        for p in glob.glob(f"{out}/jit_rcdm_step*.pt"):
            os.remove(p)
        print(f"[{name}] done in {(time.time()-t0)/60:.1f} min — saved to Drive")
    else:
        print(f"[{name}] FAILED — no {FINAL} produced; check the log above")

## 5 — The decisive diagnostic

For each arm, generate k samples from a single `h` and measure whether they
differ.  Three quantities:

- **within-h std** — spread of k samples from one `h`.  Below 1/255 = collapsed.
- **between-h std** — spread across different `h`.  The reference floor; this is
  never zero, which is exactly why the within-h number alone is uninterpretable.
- **ratio** — within/between.  Near 0 = deterministic decoder.

Also reported is the mechanism check: how much `x_pred` depends on `z` at t=0
versus t=0.5.  On the collapsed model that was 0.003 vs 0.92.

In [ ]:
import sys, torch, numpy as np
sys.path.insert(0, "/content/jit_rcdm")
sys.path.insert(0, "/content/jit_rcdm/experiments")

from common import load_generator, load_probe_encoder, resolve_device, set_seed, unnorm
from PIL import Image

DEV   = resolve_device("cuda")
FLOOR = 1.0 / 255.0
K     = 8            # samples per h
N_H   = 4            # conditioning images

# Conditioning images: reuse the packed training tensor so the notebook needs no
# extra upload.  Encoding them fresh keeps this independent of cached reps.
packed = torch.load(PACKED, weights_only=False)
imgs   = packed["images"][:N_H]                      # (N, 3, 224, 224) uint8


def diagnose(ckpt_path):
    set_seed(0)
    model, flow, cfg = load_generator(ckpt_path, DEV)
    enc, _ = load_probe_encoder("dinov3", DEV, cfg)
    S = cfg["image_size"]

    # ImageNet-normalise the packed uint8 images for the encoder.
    mean = torch.tensor([0.485, 0.456, 0.406], device=DEV).view(1, 3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225], device=DEV).view(1, 3, 1, 1)

    within, firsts = [], []
    with torch.no_grad():
        x = imgs.float().to(DEV) / 255.0
        H = enc((x - mean) / std)                    # (N_H, h_dim)

        for i in range(len(imgs)):
            noise = torch.randn(K, 3, S, S, device=DEV)
            gen = unnorm(flow.sample(model, noise, H[i:i+1].repeat(K, 1),
                                     num_steps=50, cfg_scale=1.0))
            within.append(float(gen.std(dim=0).mean()))
            firsts.append(gen[0])

        # Mechanism: does x_pred depend on z, and at which t?
        noise = torch.randn(4, 3, S, S, device=DEV)
        hh = H[0:1].repeat(4, 1)
        d = {}
        for t_val in (0.0, 0.5):
            tb = torch.full((4,), t_val, device=DEV)
            xp = model(noise, tb, hh)
            d[t_val] = float((xp[0] - xp[1]).abs().mean())

    between = float(torch.stack(firsts).std(dim=0).mean())
    w = float(np.mean(within))
    return {"within": w, "between": between,
            "ratio": w / between if between else float("nan"),
            "collapsed": w < FLOOR,
            "dxpred_t0": d[0.0], "dxpred_t05": d[0.5],
            "samples": torch.stack(firsts).cpu()}


results = {}
for name in ARMS:
    ck = local_final(name)
    if not os.path.exists(ck):
        print(f"[{name}] no checkpoint — skipped")
        continue
    print(f"\n--- {name} ---")
    results[name] = diagnose(ck)

## 6 — Verdict

In [ ]:
print(f"quantisation floor = {FLOOR:.5f}\n")
print(f"{'arm':<10} {'t_mu':>5} {'within-h':>10} {'between-h':>11} {'ratio':>7} "
      f"{'dxp t=0':>9} {'dxp t=.5':>9}  verdict")
print("-" * 84)
for name, mu in ARMS.items():
    if name not in results:
        continue
    r = results[name]
    verdict = "COLLAPSED" if r["collapsed"] else "diverse"
    print(f"{name:<10} {mu:>5.1f} {r['within']:>10.5f} {r['between']:>11.5f} "
          f"{r['ratio']:>7.3f} {r['dxpred_t0']:>9.4f} {r['dxpred_t05']:>9.4f}  {verdict}")

print()
collapsed = {n for n, r in results.items() if r["collapsed"]}
if len(results) < len(ARMS):
    print("Incomplete — train all three arms before concluding.")
elif collapsed == {"A_mu-0.8"}:
    print("HYPOTHESIS CONFIRMED: only the control collapsed. t_mu is the cause;")
    print("retrain the production model at the best-performing mu and re-run E2.")
elif len(collapsed) == len(ARMS):
    print("HYPOTHESIS REJECTED: every arm collapsed. The t-sampler is not the")
    print("lever — suspect dataset size (972 images, ~3300 epochs) or capacity.")
elif not collapsed:
    print("INCONCLUSIVE: no arm collapsed, including the control. At this budget")
    print("the model has not collapsed yet, so mu is unidentifiable. Raise")
    print("TOTAL_STEPS toward the 100k of the production run and repeat.")
else:
    print(f"PARTIAL: collapsed = {sorted(collapsed)}. Read the ratio column as a")
    print("dose-response curve rather than a yes/no.")

## 7 — Look at the samples

Numbers can hide a model that is diverse but has stopped producing retinas.
Each row is k samples from **one** `h`: a healthy arm shows visible variation
within a row while every image still looks like a fundus photograph.

In [ ]:
import matplotlib.pyplot as plt

live = [n for n in ARMS if n in results]
if live:
    fig, axes = plt.subplots(len(live), K, figsize=(1.6 * K, 1.7 * len(live)), squeeze=False)
    for row, name in zip(axes, live):
        set_seed(0)
        model, flow, cfg = load_generator(local_final(name), DEV)
        enc, _ = load_probe_encoder("dinov3", DEV, cfg)
        S = cfg["image_size"]
        mean = torch.tensor([0.485, 0.456, 0.406], device=DEV).view(1, 3, 1, 1)
        std  = torch.tensor([0.229, 0.224, 0.225], device=DEV).view(1, 3, 1, 1)
        with torch.no_grad():
            x = imgs[:1].float().to(DEV) / 255.0
            h = enc((x - mean) / std).repeat(K, 1)
            gen = unnorm(flow.sample(model, torch.randn(K, 3, S, S, device=DEV),
                                     h, num_steps=50, cfg_scale=1.0)).cpu()
        for ax, img in zip(row, gen):
            ax.imshow(img.permute(1, 2, 0).numpy())
            ax.set_xticks([]); ax.set_yticks([])
        row[0].set_ylabel(f"{name}\nt_mu={ARMS[name]}", fontsize=8)
    fig.suptitle("k samples from ONE h — variation within a row is the point", fontsize=10)
    plt.tight_layout()
    plt.show()

## 8 — Record the result

The checkpoints are already on Drive (the training cell copies each arm as it
finishes).  This writes the measurements alongside them, with the git commit and
the settings that produced them, so a number in the report can be traced to the
run behind it — the same provenance discipline the E1–E5 scripts use.

In [ ]:
import json, subprocess

commit = subprocess.run(["git", "rev-parse", "HEAD"], cwd="/content/jit_rcdm",
                        capture_output=True, text=True).stdout.strip()

record = {
    "experiment": "t_sampler_ablation",
    "hypothesis": "raising t_mu moves training mass to the clean end of the "
                  "noise-data axis and restores within-h sample diversity",
    "git_commit": commit,
    "total_steps": TOTAL_STEPS, "batch_size": BATCH_SIZE,
    "t_sigma": 0.8, "k": K, "n_h": N_H,
    "num_steps": 50, "cfg_scale": 1.0,
    "quantisation_floor": FLOOR,
    "arms": {n: {"t_mu": ARMS[n],
                 **{k: v for k, v in results[n].items() if k != "samples"}}
             for n in results},
}
with open(f"{ABL_DIR}/results.json", "w") as f:
    json.dump(record, f, indent=2)

print(f"written: {ABL_DIR}/results.json")
print(json.dumps(record["arms"], indent=2))
print("\nCheckpoints on Drive:")
for f in sorted(os.listdir(ABL_DIR)):
    print(f"  {f}")